# Metrics Utilities
Shared data structures, timing helpers, and reporting tools used by the classification and regression benchmark notebooks.

In [1]:
import time, tracemalloc
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

print("✓ Imports OK")

✓ Imports OK


## BenchmarkResult dataclass
Holds every model's metrics, timings, and optional extras (e.g. confusion matrix).

In [2]:
@dataclass
class BenchmarkResult:
    model_name: str
    task: str          # 'classification' | 'regression'
    metrics: dict[str, float] = field(default_factory=dict)
    train_time_s: float = 0.0
    predict_time_s: float = 0.0
    peak_memory_kb: float = 0.0
    extra: dict[str, Any] = field(default_factory=dict)

    def to_series(self) -> pd.Series:
        row = {
            "model": self.model_name,
            "train_time_s":    round(self.train_time_s,   4),
            "predict_time_s":  round(self.predict_time_s, 6),
            "peak_memory_kb":  round(self.peak_memory_kb, 1),
        }
        row.update({k: round(v, 4) for k, v in self.metrics.items()})
        return pd.Series(row)

# Quick smoke-test
r = BenchmarkResult(model_name="Test", task="classification", metrics={"accuracy": 0.95}, train_time_s=0.1)
print(r.to_series().to_string())

model             Test
train_time_s       0.1
predict_time_s     0.0
peak_memory_kb     0.0
accuracy          0.95


## Context managers
`timer` and `memory_tracker` wrap training / inference blocks.

In [3]:
@contextmanager
def timer():
    state = {"elapsed": 0.0}
    start = time.perf_counter()
    try:
        yield state
    finally:
        state["elapsed"] = time.perf_counter() - start

@contextmanager
def memory_tracker():
    state = {"peak_kb": 0.0}
    tracemalloc.start()
    try:
        yield state
    finally:
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        state["peak_kb"] = peak / 1024

# Demo
with memory_tracker() as mem, timer() as t:
    _ = [i**2 for i in range(500_000)]

print(f"Elapsed : {t['elapsed']:.4f}s")
print(f"Peak mem: {mem['peak_kb']:.1f} KB")

Elapsed : 1.8512s
Peak mem: 19694.9 KB


## Data helpers & reporting

In [4]:
def prepare_split(X, y, test_size=0.2, random_state=42):
    try:
        return train_test_split(X, y, test_size=test_size,
                                random_state=random_state, stratify=y)
    except ValueError:
        return train_test_split(X, y, test_size=test_size,
                                random_state=random_state)

def results_to_dataframe(results):
    return pd.DataFrame([r.to_series() for r in results]).set_index("model")

def print_summary_table(df, title="Benchmark Summary"):
    sep = "─" * 90
    print(f"\n{sep}\n  {title}\n{sep}")
    print(df.to_string())
    print(sep + "\n")

def highlight_best(df, higher_is_better, lower_is_better):
    out = df.copy().astype(str)
    for col in higher_is_better:
        if col in df.columns:
            out.loc[df[col].idxmax(), col] += " ★"
    for col in lower_is_better:
        if col in df.columns:
            out.loc[df[col].idxmin(), col] += " ★"
    return out

print("✓ All utility functions defined")

✓ All utility functions defined


## Module-level export
Other notebooks import these symbols from `metrics_utils`.

In [5]:
# Save to disk so sibling notebooks can `import metrics_utils`
import pathlib, inspect, textwrap

src = '''
import time, tracemalloc
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

@dataclass
class BenchmarkResult:
    model_name: str
    task: str
    metrics: dict = field(default_factory=dict)
    train_time_s: float = 0.0
    predict_time_s: float = 0.0
    peak_memory_kb: float = 0.0
    extra: dict = field(default_factory=dict)

    def to_series(self):
        row = {
            "model": self.model_name,
            "train_time_s": round(self.train_time_s, 4),
            "predict_time_s": round(self.predict_time_s, 6),
            "peak_memory_kb": round(self.peak_memory_kb, 1),
        }
        row.update({k: round(v, 4) for k, v in self.metrics.items()})
        return pd.Series(row)

@contextmanager
def timer():
    state = {"elapsed": 0.0}
    start = time.perf_counter()
    try:
        yield state
    finally:
        state["elapsed"] = time.perf_counter() - start

@contextmanager
def memory_tracker():
    state = {"peak_kb": 0.0}
    tracemalloc.start()
    try:
        yield state
    finally:
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        state["peak_kb"] = peak / 1024

def prepare_split(X, y, test_size=0.2, random_state=42):
    try:
        return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    except ValueError:
        return train_test_split(X, y, test_size=test_size, random_state=random_state)

def results_to_dataframe(results):
    return pd.DataFrame([r.to_series() for r in results]).set_index("model")

def print_summary_table(df, title="Benchmark Summary"):
    sep = "─" * 90
    print(f"\\n{sep}\\n  {title}\\n{sep}")
    print(df.to_string())
    print(sep + "\\n")

def highlight_best(df, higher_is_better, lower_is_better):
    out = df.copy().astype(str)
    for col in higher_is_better:
        if col in df.columns:
            out.loc[df[col].idxmax(), col] += " ★"
    for col in lower_is_better:
        if col in df.columns:
            out.loc[df[col].idxmin(), col] += " ★"
    return out
'''

pathlib.Path("c:\\Users\\SANJIT BAURI\\Desktop\\New folder (3)\\GSSoC`26\\ML-CaPsule\\Standardized Benchmarking Framework for Classification & Regression Models\\metrics_utils.py").write_text(src, encoding="utf-8")
print("✓ metrics_utils.py written")

✓ metrics_utils.py written
